# Day 8 | ML 피처 중요도 분석
## Korean Stock Undervaluation Analysis — ML Feature Importance

**목표:**
- Random Forest로 12개월 수익률 예측 시도
- SHAP으로 피처별 기여도 분석
- "어떤 변수가 수익률에 가장 영향을 미치는가?" 인사이트 도출

**결과 요약:**
```
CV R² = -0.131 (예측력 없음 → n=97개 부족, 주가 예측의 본질적 어려움)
전체 R² = 0.464 (과적합)
→ 예측보다 피처 중요도 해석에 집중

RF 피처 중요도:
1위 YoY 성장률   27.5%
2위 섹터 괴리율  22.8%
3위 섹터 PER     13.8%
4위 EPS          12.9%
5위 부채비율     10.7%
6위 시장 괴리율   6.5%  ← 핵심 지표지만 단독으로는 낮음
7위 PER           5.8%
```


## 0. 환경 세팅

In [ ]:
!pip install -q shap

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

BASE_PATH = '/content/drive/MyDrive/stock-analysis'

df_bt = pd.read_csv(f'{BASE_PATH}/data/output/backtest_result.csv',
                    dtype={'Code': str})
df_v2 = pd.read_csv(f'{BASE_PATH}/data/output/gap_v2.csv',
                    dtype={'Code': str, 'corp_code': str})

print(f"✅ df_bt: {len(df_bt)}개")


## 1. 피처 준비

In [ ]:
extra_cols = ['Code', 'yoy_growth_pct', 'debt_ratio']
df_feat = df_bt.merge(df_v2[extra_cols], on='Code', how='left')

feature_cols = [
    'gap_market_B',   # 시장 괴리율
    'gap_sector_B',   # 섹터 괴리율
    'per_B',          # PER
    'sector_per_B',   # 섹터 평균 PER
    'yoy_growth_pct', # YoY 성장률
    'debt_ratio',     # 부채비율
    'EPS',            # 주당순이익
]

df_ml = df_feat[feature_cols + ['ret_B_12m', 'Name']].dropna()
X = df_ml[feature_cols].values
y = df_ml['ret_B_12m'].values

print(f"✅ ML 데이터셋: {len(df_ml)}개 / 피처: {len(feature_cols)}개")


## 2. Random Forest 학습

In [ ]:
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=4,
    min_samples_leaf=5,
    random_state=42
)

# 5-Fold CV
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='r2')
print(f"5-Fold CV R²: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")

# 전체 학습 (SHAP용)
rf.fit(X, y)
print(f"전체 R²: {r2_score(y, rf.predict(X)):.3f}")

feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n── RF 피처 중요도 ──")
for _, row in feat_imp.iterrows():
    bar = '█' * int(row['importance'] * 50)
    print(f"  {row['feature']:<20} {row['importance']:.3f} {bar}")


## 3. SHAP 분석

In [ ]:
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X)

mean_shap = np.abs(shap_values).mean(axis=0)
shap_imp  = pd.DataFrame({
    'feature':   feature_cols,
    'mean_shap': mean_shap
}).sort_values('mean_shap', ascending=False)

print("── SHAP 피처 중요도 ──")
for _, row in shap_imp.iterrows():
    bar = '█' * int(row['mean_shap'] / mean_shap.max() * 20)
    print(f"  {row['feature']:<20} {row['mean_shap']:.2f} {bar}")


## 4. 저장

In [ ]:
feat_imp.to_csv(f'{BASE_PATH}/data/output/rf_feature_importance.csv',
               index=False, encoding='utf-8-sig')
shap_imp.to_csv(f'{BASE_PATH}/data/output/shap_importance.csv',
                index=False, encoding='utf-8-sig')
print("✅ rf_feature_importance.csv 저장 완료")
print("✅ shap_importance.csv 저장 완료")
